# MVP pipeline imágenes de repuestos (DONREP)

## Etapa 0 — Carga de datos

Librerias 

In [8]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

INPUT_XLSX = ROOT / "input" / "products.xlsx"
COL_REF, COL_BRAND, COL_NAME = "Ref Proveedor", "Marca", "Nombre"

df = pd.read_excel(INPUT_XLSX, dtype={COL_REF: str})
df = df[[COL_REF, COL_BRAND, COL_NAME]].copy()
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()
df = df[df[COL_REF].notna() & (df[COL_REF] != "") & (df[COL_REF] != "nan")].reset_index(drop=True)

print(f"{len(df)} productos cargados")
df.head(10)

281 productos cargados


,Ref Proveedor,Marca,Nombre
0,8550504748,MOTRIO,10W30 MOTRIO 1LX12UN
1,8550504747,MOTRIO,10W30 MOTRIO 4LX4UND
2,7711737740,MOTRIO,10W30 MOTRIO 4X4 UNID
3,8550504765,MOTRIO,10W40 MOTRIO A3/B4 SS 1LX12UND
4,8550504764,MOTRIO,10W40 MOTRIO A3/B4 SS 4LX4UND
5,8550504745,MOTRIO,15W40 MOTRIO 1LX12UN
6,7711737737,MOTRIO,15W40 MOTRIO 4LX4UND
7,8550504744,MOTRIO,15W40 MOTRIO 4LX4UND
8,7711649209,RENAULT,185/65R15 ENERGY XM2
9,8550504751,MOTRIO,20W50 MOTRIO 1LX12UN


## Etapa 1 — Normalización de nombres

Es importante revisar visualmente `nombre_original` vs `nombre_limpio` / `termino_busqueda` luego de la normalizacion. De ser necesario se puede ajustar `config/abbreviations.py` .

In [10]:
from pipeline.normalize import normalize_df

df_norm = normalize_df(df, col_nombre=COL_NAME, col_marca=COL_BRAND)

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 300)
df_norm[["nombre_original", "nombre_limpio", "termino_busqueda"]].head(10)

,nombre_original,nombre_limpio,termino_busqueda
0,10W30 MOTRIO 1LX12UN,10W30 MOTRIO 1L,10W30 MOTRIO 1L
1,10W30 MOTRIO 4LX4UND,10W30 MOTRIO 4L,10W30 MOTRIO 4L
2,10W30 MOTRIO 4X4 UNID,10W30 MOTRIO 4X4 UNID,10W30 MOTRIO 4X4 UNID
3,10W40 MOTRIO A3/B4 SS 1LX12UND,10W40 MOTRIO A3/B4 SS 1L,10W40 MOTRIO A3/B4 SS 1L
4,10W40 MOTRIO A3/B4 SS 4LX4UND,10W40 MOTRIO A3/B4 SS 4L,10W40 MOTRIO A3/B4 SS 4L
5,15W40 MOTRIO 1LX12UN,15W40 MOTRIO 1L,15W40 MOTRIO 1L
6,15W40 MOTRIO 4LX4UND,15W40 MOTRIO 4L,15W40 MOTRIO 4L
7,15W40 MOTRIO 4LX4UND,15W40 MOTRIO 4L,15W40 MOTRIO 4L
8,185/65R15 ENERGY XM2,185/65R15 ENERGY XM2,185/65R15 ENERGY XM2 RENAULT
9,20W50 MOTRIO 1LX12UN,20W50 MOTRIO 1L,20W50 MOTRIO 1L


In [11]:
# columnas estructuradas extraídas
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].head(10)

,nombre_original,pack,litraje,pack_ambiguo,viscosidad,medida_llanta,vehiculo_code_raw
0,10W30 MOTRIO 1LX12UN,1LX12UN,1L,False,10W30,,
1,10W30 MOTRIO 4LX4UND,4LX4UND,4L,False,10W30,,
2,10W30 MOTRIO 4X4 UNID,4X4 UNID,,True,10W30,,UNID
3,10W40 MOTRIO A3/B4 SS 1LX12UND,1LX12UND,1L,False,10W40,,
4,10W40 MOTRIO A3/B4 SS 4LX4UND,4LX4UND,4L,False,10W40,,
5,15W40 MOTRIO 1LX12UN,1LX12UN,1L,False,15W40,,
6,15W40 MOTRIO 4LX4UND,4LX4UND,4L,False,15W40,,
7,15W40 MOTRIO 4LX4UND,4LX4UND,4L,False,15W40,,
8,185/65R15 ENERGY XM2,,,False,,185/65R15,XM2
9,20W50 MOTRIO 1LX12UN,1LX12UN,1L,False,20W50,,


In [12]:
df_norm[["nombre_original", "pack", "litraje", "pack_ambiguo", "viscosidad", "medida_llanta", "vehiculo_code_raw"]].tail(10)

,nombre_original,pack,litraje,pack_ambiguo,viscosidad,medida_llanta,vehiculo_code_raw
271,VIDRIO LUNETA TR SA3,,,False,,,SA3
272,VIDRIO MOVIL PUERTA NDU,,,False,,,NDU
273,VIDRIO MOVIL PUERTA TRASERA DERECHA X52,,,False,,,X52
274,VIDRIO MOVIL PUERTA TRASERA IZQUIER GKO,,,False,,,GKO
275,VIDRIO PARABRISA NDU,,,False,,,NDU
276,VIDRIO PARABRISAS NKG,,,False,,,NKG
277,VIDRIO PARABRISAS OR2,,,False,,,OR2
278,VIDRIO PARABRISAS SIN SENSOR LLUVIA NDU,,,False,,,NDU
279,VIDRIO PTA TR IZ SA3,,,False,,,SA3
280,VIDRIO PUERTA DELANTERA DERECHA NDUH,,,False,,,NDUH


### Filas de pack ambiguo (revisar a mano)

Tienen expresión de pack pero sin "L" explícito, así que no se pudo separar litraje de multiplicador de caja - quedaron intactas en `nombre_limpio`. ¿`4X4 UNID` es un típo de `4LX4UND`? ¿Qué significan `3X5` / `6X1` de Castrol?

In [14]:
df_norm[df_norm["pack_ambiguo"]][["nombre_original", "nombre_limpio", "pack"]]

,nombre_original,nombre_limpio,pack
2,10W30 MOTRIO 4X4 UNID,10W30 MOTRIO 4X4 UNID,4X4 UNID
24,AMOR TRA MOT 4X4 DU,AMORTIGUADOR TRASERO MOTOR 4X4 DU,4X4
65,CASTROL 10W40 3X5,CASTROL 10W40 3X5,3X5
66,CASTROL 10W40 6X1,CASTROL 10W40 6X1,6X1
67,CASTROL 10W40 6X1,CASTROL 10W40 6X1,6X1
68,CASTROL 15W40 3X5,CASTROL 15W40 3X5,3X5
69,CASTROL 15W40 3X5,CASTROL 15W40 3X5,3X5
70,CASTROL 15W40 6X1,CASTROL 15W40 6X1,6X1
71,CASTROL 15W40 6X1,CASTROL 15W40 6X1,6X1
72,CASTROL 20W50 3X5,CASTROL 20W50 3X5,3X5


## Etapa 2 — Categorización de productos, con base en los nombres


In [16]:
from pipeline.categorize import categorize_df, coverage_report
coverage_report(df_norm)          # prints per-category counts + the `otros` backlog list
cd = categorize_df(df_norm)       # adds categoria / cse_profile / presentacion columns
cd[["nombre_limpio", "categoria", "cse_profile"]].head(10)

Cobertura: 281 filas, 14 categorias

   58  carroceria
   37  vidrios_espejos
   33  iluminacion
   29  lubricantes
   21  suspension_direccion
   20  merchandising
   13  motor_transmision
   13  otros
   13  emblemas
   12  frenos
   10  filtros
    8  refrigeracion
    7  llantas_rines
    7  baterias

'otros' (backlog para nuevas reglas): 13 (5%)
    - CAMPANA ABS SA2-LO2 "48P"
    - GRUPO MOTOVENTILADOR REF H5D KD
    - JGO PLUMILLAS LN3
    - JUEGO PASTILLAS FRENO TRASERO ZO2
    - JUEGO PASTILLAS FRENOS DELANTEROS NDUH
    - JUEGO PASTILLAS FRENOS TRASERO NDUH BOR
    - KIT JUNTA DECANTADOR NT2
    - KIT JUNTAS INDICADOR CARBURANTE
    - MODULO VARIA CLIM NM
    - PASTILLAS FRENOS DELANTEROS KW
    - PLASTIGLO 250ML TT
    - TAPETE TERMOFORMADO MILDH NDUH
    - TAPETE TERMOFORMADO NDUH


,nombre_limpio,categoria,cse_profile
0,10W30 MOTRIO 1L,lubricantes,white_dominant
1,10W30 MOTRIO 4L,lubricantes,white_dominant
2,10W30 MOTRIO 4X4 UNID,lubricantes,white_dominant
3,10W40 MOTRIO A3/B4 SS 1L,lubricantes,white_dominant
4,10W40 MOTRIO A3/B4 SS 4L,lubricantes,white_dominant
5,15W40 MOTRIO 1L,lubricantes,white_dominant
6,15W40 MOTRIO 4L,lubricantes,white_dominant
7,15W40 MOTRIO 4L,lubricantes,white_dominant
8,185/65R15 ENERGY XM2,llantas_rines,baseline
9,20W50 MOTRIO 1L,lubricantes,white_dominant
